# clock/01.1 — datatype sanity check

Everything in this line of work — the goal-head mask, the completion (move-on) mask, and both
aggregated vectors — was derived on the **`int`** corpus. This notebook asks whether the same
machinery still behaves on the other datatypes, **qualitatively**, on a handful of samples. No
scores, no plots; the point is to look at the generations before spending a full eval on them.

Set `DATATYPE` in §0 and run top to bottom.

1. **§1 — does the LM behave?** Load 5–6 samples for the datatype, show the prompt, greedily
   generate, and check the obvious things: one variable per condition, letters in order, code that
   parses and passes its own unit tests.
2. **§2 — GoalScope.** Reuses `scripts.evaluation.eval_goalscope`'s **own** functions
   (`build_corpus` → `build_prompt` → `generate_completion` → `rule_based_spans` →
   `anchors_for_sample` → `apply_goalscope`), so this cannot drift from what the script actually
   measures. Read-outs are printed against the implementation span they were taken from.
3. **§3 — move-on.** The `16_move_on_signal` setup: a prompt pivoted mid-implementation of
   `target_slot >= 1` (i.e. `b` or later), with at least one **earlier** condition force-prefilled
   as a multi-line block so the multi-line precedent is set. Clean vs patched, side by side.

Read §1 first: if the model does not produce well-formed `a = …` / `b = …` code for this datatype,
§2's anchors and §3's pivot are both meaningless and nothing below is interpretable.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys, re, io, json, random, logging, contextlib
sys.path.append("../..")

import numpy as np
import torch
from rich.console import Console, Group
from rich.theme import Theme
from rich.table import Table
from rich.panel import Panel
from rich.text import Text
from rich.box import SIMPLE_HEAVY
from rich.markup import escape
from rich.syntax import Syntax
from src.utils import logging_utils, env_utils

logging_utils.configure(level=logging.WARNING)
logger = logging.getLogger(__name__)
console = Console(theme=Theme({"default": "white"}), highlight=False)
def rich_print(*a, **k): console.print(*a, **k)
RES = env_utils.DEFAULT_RESULTS_DIR

In [2]:
from src.models import ModelandTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-32B-Instruct"
mt = ModelandTokenizer(model_key=MODEL_NAME, dtype=torch.bfloat16,
                       device_map="auto", attn_implementation="eager")
N_LAYER, N_HEADS = mt.n_layer, mt.model.config.num_attention_heads
print(f"n_layer={N_LAYER}  n_heads={N_HEADS}")

2026-08-13 15:04:23 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

n_layer=64  n_heads=40


In [3]:
# ── §0: what to check ───────────────────────────────────────────────────────
from src.goals.condition_data import DATATYPES, predicates_for

DATATYPE   = "list[str]"          # "int" | "str" | "list[int]" | "list[str]"
N_SAMPLES  = 8
N_COND     = [3, 4]         # conditions per spec
SEED       = 123
GEN_TOKENS = 256            # completion budget for the full-program generation

_pool = predicates_for(DATATYPE)
_spec = DATATYPES[DATATYPE]
# escape(): rich would parse the "[int]" in "list[int]" as a style tag and silently drop it
rich_print(f"[bold]DATATYPE[/bold] = {escape(repr(DATATYPE))}   ·   arg [yellow]{_spec.arg}[/yellow] "
           f"({escape(_spec.annotation)})   ·   {len(_pool)} predicates in the bank")
if DATATYPE != "int":
    rich_print("[grey58]OOD: every mask and cached vector below was derived on the `int` corpus, "
               "so a weak result here is ambiguous between 'the signal does not generalise' and "
               "'these particular heads do not'.[/grey58]")
print("  " + ", ".join(p.id for p in _pool))

DATATYPE = 'list[str]'   ·   arg words (list[str])   ·   10 predicates in the bank

OOD: every mask and cached vector below was derived on the `int` corpus, so a weak result here is ambiguous between
'the signal does not generalise' and 'these particular heads do not'.

  ls_len_ge, ls_all_lower, ls_total_chars_gt, ls_longest_ge, ls_distinct, ls_sorted_alpha, ls_same_length, ls_starts_vowel_ge, ls_has_long_word, ls_shares_initial


In [4]:
# ── §1: does the LM behave on this datatype? ────────────────────────────────
from scripts.evaluation.eval_goalscope import build_corpus, build_prompt, generate_completion

SAMPLES = build_corpus(N_SAMPLES, N_COND, SEED, DATATYPE)
GEN = []                                   # (sample, prompt, completion, full_text)
for s in SAMPLES:
    p = build_prompt(mt, s)
    c = generate_completion(mt, p, GEN_TOKENS)
    GEN.append(dict(sample=s, prompt=p, completion=c, full_text=p + c))
print(f"{len(GEN)} generations for datatype={DATATYPE!r}")

def code_of(g):
    """The generated function body, cut at the first fence/EOS the model emits."""
    return g["completion"].split("</code>")[0].split("```")[0].rstrip()

def behaviour_row(g):
    """The three things §2 and §3 depend on: letters in order, one per condition, runnable code."""
    n = len(g["sample"].spec.predicate_ids)
    want = [chr(97 + k) for k in range(n)]
    code = g["sample"].render_code_stub() + code_of(g)
    found = []
    for m in re.finditer(r"(?m)^[ \t]+([a-z])\s*=(?!=)", code):
        if m.group(1) not in found:
            found.append(m.group(1))
    ok_letters = found[:n] == want
    try:
        compile(code, "<gen>", "exec"); ok_parse = True
    except SyntaxError:
        ok_parse = False
    # does it actually compute the conditions correctly? run the spec's own unit tests
    passed = total = 0
    if ok_parse:
        # the generated function ends in a print(); swallow it, or 6 samples x 6 inputs of
        # unrelated output buries the table
        for x in g["sample"].spec.test_inputs:
            ns = {}
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    exec(code, ns)
                    ns["check_conditions"](x)
                passed += 1
            except Exception:
                pass
            total += 1
    return dict(spec=g["sample"].spec.spec_id, n_cond=n, letters="".join(found[:n]),
                ok_letters=ok_letters, ok_parse=ok_parse, runs=f"{passed}/{total}")

t = Table(title=f"LM behaviour on datatype={DATATYPE!r}", box=SIMPLE_HEAVY, pad_edge=False,
          title_justify="left")
for c_ in ("spec", "n_cond", "letters", "a,b,c… in order", "parses", "runs w/o error"):
    t.add_column(c_, justify="right")
ROWS1 = [behaviour_row(g) for g in GEN]
for r in ROWS1:
    t.add_row(r["spec"], str(r["n_cond"]), r["letters"] or "—",
              "[green]yes[/green]" if r["ok_letters"] else "[red]NO[/red]",
              "[green]yes[/green]" if r["ok_parse"] else "[red]NO[/red]", r["runs"])
rich_print(t)
rich_print("[grey58]`runs w/o error` executes the generated function on the spec's own test inputs "
           "— it checks that the code is callable, NOT that the conditions are correct.[/grey58]")

for g in GEN[:2]:
    instr = g["prompt"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()
    rich_print(Panel(Group(Text(instr, style="grey70"), Text(""),
                           Syntax(g["sample"].render_code_stub() + code_of(g), "python",
                                  theme="monokai", word_wrap=True)),
                     title=f"[bold]{g['sample'].spec.spec_id}[/bold]", title_align="left",
                     border_style="cyan", padding=(1, 1)))

8 generations for datatype='list[str]'


LM behaviour on datatype='list'                                              
                                                                             
         spec   n_cond   letters   a,b,c… in order   parses   runs w/o error 
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 eval_list_00        3       abc               yes      yes              6/6 
 eval_list_01        4      abcd               yes      yes              6/6 
 eval_list_02        3       abc               yes      yes              6/6 
 eval_list_03        3       abc               yes      yes              6/6 
 eval_list_04        3       abc               yes      yes              6/6 
 eval_list_05        4      abcd               yes      yes              6/6 
 eval_list_06        4      abcd               yes      yes              6/6 
 eval_list_07        4      abcd               yes      yes              6/6

`runs w/o error` executes the generated function on the spec's own test inputs — it checks that the code is 
callable, NOT that the conditions are correct.

╭─ eval_list_00 ──────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a Python program that implements a function `check_conditions` which, given a list of strings words,      │
│ checks the following conditions:                                                                                │
│                                                                                                                 │
│ 1. all words in words are distinct                                                                              │
│ 2. every word in words is entirely lowercase                                                                    │
│ 3. all words in words have the same length                                                                      │
│                                                                                                                 │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable,      │
│ using single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all │
│ conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each    │
│ condition passed individually and whether all of them passed collectively.                                      │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.                                                                                    │
│ Do not include any comments in your code.                                                                       │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a=len(words)==len(set(words))                                                                               │
│     b=all(w.islower() for w in words)                                                                           │
│     c=len(set(len(w) for w in words))==1                                                                        │
│     print(f"Distinct: {a}, Lowercase: {b}, Same Length: {c}, All: {a and b and c}")                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ eval_list_01 ──────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a Python program that implements a function `check_conditions` which, given a list of strings words,      │
│ checks the following conditions:                                                                                │
│                                                                                                                 │
│ 1. all words in words are distinct                                                                              │
│ 2. at least 2 words in words start with a vowel                                                                 │
│ 3. words has at least 4 entries                                                                                 │
│ 4. all words in words have the same length                                                                      │
│                                                                                                                 │
│ For each condition, in the order listed, assign the boolean result of the check to its own local variable,      │
│ using single lowercase letters in order — `a` for the first condition, `b` for the second, and so on. After all │
│ conditions have been checked, use a SINGLE print statement that, using those variables, reports whether each    │
│ condition passed individually and whether all of them passed collectively.                                      │
│ Compute each condition's boolean result inside `check_conditions` (one local variable per condition); do not    │
│ define any helper functions.                                                                                    │
│ Do not include any comments in your code.                                                                       │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = len(words) == len(set(words))                                                                           │
│     b = sum(1 for word in words if word[0] in 'aeiouAEIOU') >= 2                                                │
│     c = len(words) >= 4                                                                                         │
│     d = len(set(len(word) for word in words)) == 1                                                              │
│     print(f"a: {a}, b: {b}, c: {c}, d: {d}, all: {a and b and c and d}")                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 2. GoalScope

Exactly the script's path — `rule_based_spans` for the per-condition implementation text,
`anchors_for_sample` for the five token positions per line, then one `apply_goalscope` call
restricted to those positions.

`MODE = "faithfulness"` steers into `apply_goalscope`'s default NL probe ("what are you doing in
this line?"); `"causality"` steers into the information-less C++ stub `bool check(auto value){` and
the read-out is the C++ body it then writes.

In [5]:
# ── §2: GoalScope read-outs at every anchor ─────────────────────────────────
from scripts.evaluation.eval_goalscope import (
    ANCHOR_ORDER, CAUSAL_PREFIX, anchors_for_sample, build_causality_prompt,
    default_goal_band, load_head_set, rule_based_spans,
)
from src.goals.goalscope import apply_goalscope

MODE       = "faithfulness"      # "faithfulness" | "causality"
HEADS_SPEC = "goal"              # "goal" | "gaze" | "decision" | <path>.json
STRATEGY   = "additive"
ALPHA      = 0.2
STEER_BAND = (35, 46)
GS_MAX_NEW = 64
N_SHOW     = 3                   # samples to print read-outs for

HEADS = load_head_set(HEADS_SPEC, MODEL_NAME, RES, N_HEADS, goal_threshold=0.9,
                      goal_band=None, n_layer=N_LAYER)
gs_kwargs = dict(selected_heads=HEADS, strategy=STRATEGY, alpha=ALPHA,
                 steer_layer_range=STEER_BAND, n_steer_tokens="all",
                 max_new_tokens=GS_MAX_NEW)
if MODE == "causality":
    gs_kwargs["goalscope_prompt"] = build_causality_prompt(mt)
READOUT_PREFIX = CAUSAL_PREFIX if MODE == "causality" else ""
print(f"{len(HEADS)} {HEADS_SPEC} heads · {STRATEGY} α={ALPHA} · band {STEER_BAND} · mode={MODE} "
      f"· goal band {default_goal_band(MODEL_NAME, n_layer=N_LAYER)}")

GS = []
for g in GEN[:N_SHOW]:
    conditions = list(g["sample"].spec.spec_texts)
    impl_spans = rule_based_spans(g["full_text"], len(g["prompt"]),
                                  len(g["sample"].spec.predicate_ids))
    rows = anchors_for_sample(mt, g["full_text"], g["prompt"], conditions, impl_spans)
    apos = sorted({p for r in rows for p in r["anchors"].values()})
    if not apos:
        rich_print(f"[red]{g['sample'].spec.spec_id}: no anchors found — "
                   f"the generation is not `    {{var}} = …` shaped[/red]")
        continue
    res = apply_goalscope(g["full_text"], mt=mt, show=False, include_noncode=False,
                          token_positions=apos, **gs_kwargs)
    readouts = {r["pos"]: r for r in res}
    GS.append(dict(spec=g["sample"].spec.spec_id, rows=rows, readouts=readouts))

    for row in rows:
        head = (f"[bold]{g['sample'].spec.spec_id}[/bold] · {row['var']} · "
                f"[yellow]{escape(row['condition'])}[/yellow]\n"
                f"[grey70]implementation: {escape(row['impl_span']) or '(none found)'}[/grey70]")
        t = Table(box=SIMPLE_HEAVY, expand=True, pad_edge=False, show_header=True)
        t.add_column("anchor", justify="right", no_wrap=True)
        t.add_column("read-out", ratio=1, overflow="fold", style="sea_green3")
        for aname in ANCHOR_ORDER:
            if aname not in row["anchors"]:
                continue
            ro = READOUT_PREFIX + readouts.get(row["anchors"][aname], {}).get("patched_gen", "")
            t.add_row(aname, escape(ro.strip()) or "[grey42](empty)[/grey42]")
        rich_print(Panel(Group(Text.from_markup(head), Text(""), t),
                         border_style="grey42", padding=(1, 1)))

122 goal heads · additive α=0.2 · band (35, 46) · mode=faithfulness · goal band (25, 56)


GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
You have set `compile_config`, but we are unable to meet the criteria for compilation. Compilation will be skipped.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_00 · a · all words in words are distinct                                                              │
│ implementation: a=len(words)==len(set(words))                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm executing the line of code and determining the state of variables based on that execution.    │
│               Here's how it would look:                                                                         │
│                                                                                                                 │
│               ```python                                                                                         │
│               a = 5                                                                                             │
│               b = a == 3                                                                                        │
│               c = b and a != 4                                                                                  │
│               d = c or b is False                                                                               │
│               ```                                                                                               │
│                                                                                                                 │
│               Each letter represents a step                                                                     │
│      indent   Sure, I'll check.                                                                                 │
│                                                                                                                 │
│               ```python                                                                                         │
│               a = len(set(s)) == len(s)                                                                         │
│               b = all(i == j for i, j in zip(a, (len(set(s)), len(set(s)) == len(s), len(set(s)) == len(s) and  │
│               len(set(s)) == len(s)))                                                                           │
│               c                                                                                                 │
│         var   Sure, let's break it down:                                                                        │
│                                                                                                                 │
│               1. `a = len(set([x for x in [True, False] if x == (len(set([x for x in [True, False] if x ==      │
│               (len(set([x for x in [True, False] if x == (len(set([                                             │
│          eq   Checking for uniqueness using a set.                                                              │
│  first_impl   Checking for uniqueness using a set.                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_00 · b · every word in words is entirely lowercase                                                    │
│ implementation: b=all(w.islower() for w in words)                                                               │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm checking if the variable `a` is true.                                                         │
│      indent   I'm checking if all characters in the string are lowercase.                                       │
│               I'm verifying if all characters are unique by comparing the length of the set of the string to    │
│               the original string's length.                                                                     │
│               I'm ensuring there are no uppercase letters by comparing the original string to its lowercase     │
│               version.                                                                                          │
│         var   I'm checking if all characters in the string are lowercase.                                       │
│          eq   I'm checking if all characters in the string are lowercase.                                       │
│  first_impl   I'm checking if all characters in the string are lowercase.                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_00 · c · all words in words have the same length                                                      │
│ implementation: c=len(set(len(w) for w in words))==1                                                            │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm checking if all elements in the list are the same.                                            │
│               ```python                                                                                         │
│               all_same = len(set(lst)) == 1                                                                     │
│               ```                                                                                               │
│               I'm verifying that the length of the set created from the list is 1, ensuring all elements have   │
│               the same length.                                                                                  │
│               ```python                                                                                         │
│               all_equal_length = len(set(map                                                                    │
│      indent   Checking if all elements in the list have the same length.                                        │
│         var   Checking if all elements in the list have the same length.                                        │
│          eq   Checking if all elements have the same length by comparing the length of the first element to     │
│               the length of the rest.                                                                           │
│  first_impl   I'm checking if all elements have the same length by comparing the length of the first element    │
│               to the length of the rest.                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (5 outputs left out)

GoalScope[additive]:   0%|          | 0/1 [00:00<?, ?it/s]

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_02 · a · all words in words are distinct                                                              │
│ implementation: a = len(words) == len(set(words))                                                               │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm analyzing the code and determining what each line does, then responding accordingly. Here's   │
│               how it would look:                                                                                │
│                                                                                                                 │
│               ```python                                                                                         │
│               def check_code():                                                                                 │
│                   a = True                                                                                      │
│                   b = False                                                                                     │
│                   c = a == b                                                                                    │
│                   d = c != (a and b)                                                                            │
│                   e = d or not c                                                                                │
│      indent   Sure, let's assume `code` is the variable holding the code:                                       │
│                                                                                                                 │
│               ```python                                                                                         │
│               a = len(code) == len(set(code))                                                                   │
│               b = all(i in 'aeiou' for i in code)                                                               │
│               c = len(set(code)) == len([*map(lambda x: x.isalpha(), code)])                                    │
│               d =                                                                                               │
│         var   Sure, let's assume the line is checking if a variable `a` is true for uniqueness:                 │
│               ```python                                                                                         │
│               a = len(set([x for x in [len(set([y for y in [a]])])])) == 1                                      │
│               ```                                                                                               │
│               This line checks if all elements in the list are unique and assigns                               │
│          eq   Checking if all elements in a list are unique by comparing the length of the list to the length   │
│               of a set created from the list.                                                                   │
│  first_impl   Checking for uniqueness using a set to ensure all elements are distinct.                          │
│                                                                                                                 │
│                                                       

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_02 · b · at least 3 words in words start with a vowel                                                 │
│ implementation: b = sum(1 for word in words if word[0].lower() in 'aeiou') >= 3                                 │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm checking if the variable `a` is unique by converting it to a set and comparing its length to  │
│               1, then assigning the result to `b`.                                                              │
│      indent   Sure, I'll check for vowels in the first word: `if len([c for c in word if c.lower() in           │
│               'aeiou']) >= 1 and len([w for w in words if w[0].lower() in 'aeiou']) >= 1 and len([v for v in    │
│               words                                                                                             │
│         var   Sure, `len([x for x in my_list if x]) >= 3`                                                       │
│          eq   Sure, `sum(1 for c in s if c.lower() in 'aeiou' and c.lower().count('a') >= 3) >= 3`              │
│  first_impl   Checking if the first character of each string in a list is a vowel.                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ eval_list_02 · c · the longest word in words has at least 6 characters                                          │
│ implementation: c = max((len(word) for word in words), default=0) >= 6                                          │
│                                                                                                                 │
│                                                                                                                 │
│      anchor   read-out                                                                                          │
│  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│    prev_end   I'm checking if the current line is the last one in the code.                                     │
│      indent   I'm checking if the length of the longest word is greater than the total number of words.         │
│         var   I'm checking if the current line is the last one, evaluating if all conditions have been met.     │
│          eq   I'm checking if the length of the longest string in the list is greater than or equal to 6.       │
│  first_impl   I'm checking the length of the longest string in a list and comparing it to a specific value.     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## 3. Move-on intervention

Same construction as `16_move_on_signal`: `build_example(..., style="multiline")` renders a spec
with the target condition pivoted mid-implementation. `target_slot >= 1` is required by
`build_example` and is what we want anyway — it guarantees at least one **earlier** condition is
force-prefilled as a completed multi-line block, which sets the precedent that a condition may span
several lines. So the pivot is always on `b` or later.

The move-on vector is the mean decision-head output at the pivot over `sl` (single-line, i.e.
already-finished) contexts, exactly as in `16` §1 — cached **on this datatype**, so §3 tests whether
the *heads* transfer, holding the vector in-distribution.

In [6]:
# ── §3: move-on at a pivot in slot >= 1 ("b" or later) ──────────────────────
from src.goals.head_loc_attn import HeadMask, get_o_proj_inputs_batch
from src.goals.completion_pairs import build_completion_pairs, n_multiline_preds
from src.goals.goalscope import headpatched_generate
from src.goals.oid_gaze import cache_root
from src.tokens import prepare_input

VERSION, DEC_TAG, DEC_THR, MO_ALPHA = "conditions", "lam10.0", 0.8, 2.0
MO_SLOTS   = [1, 2, 3]            # 1 -> pivot on `b`, 2 -> `c`, …  (>=1 by construction)
MO_N_SRC   = 64                   # source contexts for the mean vector
MO_BATCH   = 16
MO_SHOW    = 5                    # qualitative pairs to print
MO_MAX_NEW = 48

_root = os.path.join(cache_root(MODEL_NAME, VERSION), "completion_mask")
_c = [t for t in os.listdir(_root) if os.path.isdir(os.path.join(_root, t)) and DEC_TAG in t]
assert _c, f"no completion_mask with tag {DEC_TAG!r} under {_root}"
DCK = torch.load(os.path.join(_root, max(_c, key=lambda t: os.path.getmtime(os.path.join(_root, t))),
                              "mask.pt"), map_location="cpu", weights_only=False)
dec_layers = [int(x) for x in DCK["layers"]]
decision_heads = [(dec_layers[li], int(h))
                  for li, h in np.argwhere(DCK["mask_values"].numpy() > DEC_THR)]
PATCH_POS = int(DCK["patch_tokens"][0]) if len(DCK.get("patch_tokens", [-2])) == 1 else -2

def mask_bin(heads):
    m = HeadMask(N_LAYER, N_HEADS)
    with torch.no_grad():
        m.values.zero_()
        for (l, h) in heads: m.values[l, h] = 1.0
    return m
DEC_MASK, NULL_MASK = mask_bin(decision_heads), mask_bin([])

# pairs are built on THIS datatype, so the cached vector is in-distribution and only the HEADS
# are being asked to transfer
PAIRS = []
for s in MO_SLOTS:
    PAIRS += build_completion_pairs(mt, datatype=DATATYPE, n_pairs=MO_N_SRC, target_slots=(s,),
                                    direction="sl->ml", same=True, patch_tokens=(PATCH_POS,),
                                    seed=1234 + s, log=logger.debug)
assert PAIRS, (f"no completion pairs for datatype={DATATYPE!r} — the pool has "
               f"{n_multiline_preds(DATATYPE)} multiline predicates; try fewer MO_SLOTS")

_acc = {l: [] for l in dec_layers}
_src = [p.source.sl_prompt for p in PAIRS]          # sl = the finished (move-on) rendering
for i in range(0, len(_src), MO_BATCH):
    cache = get_o_proj_inputs_batch(mt, _src[i:i + MO_BATCH], dec_layers,
                                    token_positions=[PATCH_POS])
    for l in dec_layers: _acc[l].append(cache[l][:, 0, :].float().cpu())
MEAN_SRC = torch.stack([torch.cat(_acc[l], 0).mean(0) for l in dec_layers])
print(f"decision heads {len(decision_heads)} · PATCH_POS={PATCH_POS} · "
      f"{len(PAIRS)} pairs on {DATATYPE!r} · vector {tuple(MEAN_SRC.shape)}")

def render_spec(ctx):
    """The spec as the model was told it, with the pivot and the move-on target marked.

    `target_slot` is the condition being written when the pivot lands, so its implementation is
    half-finished; `target_slot + 1` is where a successful move-on should go. Everything before is
    already complete in the prefill (and slot < target_slot is what sets the multi-line precedent)."""
    ts = ctx.target_slot
    lines = []
    for k, text in enumerate(ctx.spec.spec_texts):
        var = chr(97 + k)
        if k < ts:
            lines.append(f"[grey50]  {var}. {escape(text)}   (already written)[/grey50]")
        elif k == ts:
            lines.append(f"[yellow]  {var}. {escape(text)}   <- CURRENT: pivot is mid-implementation"
                         f"[/yellow]")
        elif k == ts + 1:
            lines.append(f"[salmon1]  {var}. {escape(text)}   <- TARGET: move-on should land here"
                         f"[/salmon1]")
        else:
            lines.append(f"[grey62]  {var}. {escape(text)}[/grey62]")
    return "\n".join(lines)

for p in PAIRS[:MO_SHOW]:
    ctx = p.source
    prompt = ctx.ml_prompt                         # ml = pivoted mid-implementation -> should STAY
    inp = prepare_input(prompts=prompt, tokenizer=mt)
    clean = headpatched_generate(mt, dict(inp), MEAN_SRC, NULL_MASK, dec_layers,
                                 max_new_tokens=MO_MAX_NEW, patch_input_positions=[PATCH_POS])
    patched = headpatched_generate(mt, dict(inp), MEAN_SRC, DEC_MASK, dec_layers,
                                   max_new_tokens=MO_MAX_NEW, n_steer_tokens=1,
                                   patch_input_positions=[PATCH_POS], alpha=MO_ALPHA)
    body = prompt.split("<code>")[-1].replace("    \n", "\n").rstrip()
    ts = ctx.target_slot
    cur_v, tgt_v = chr(97 + ts), chr(97 + ts + 1)
    # ml_ans / sl_ans are the two argmax next-tokens the pair was built from: the stay answer and
    # the move-on answer. Decoding them says what "correct" looks like for this prompt.
    stay_tok = mt.tokenizer.decode([ctx.ml_ans]).strip()
    move_tok = mt.tokenizer.decode([ctx.sl_ans]).strip()
    rich_print(Panel(
        Group(Text.from_markup(
                  f"[bold]{escape(ctx.spec.spec_id)}[/bold]  ·  datatype "
                  f"[bold]{escape(DATATYPE)}[/bold]  ·  target_slot={ts}\n"
                  f"pivot on [yellow]{cur_v}[/yellow]   ->   move-on target "
                  f"[salmon1]{tgt_v}[/salmon1]      "
                  f"[grey58](stay token {stay_tok!r} · move-on token {move_tok!r})[/grey58]"),
              Text(""),
              Text.from_markup(render_spec(ctx)),
              Text(""),
              Text("prefill (pivot = end of the last line):", style="bold"),
              Syntax(body, "python", theme="monokai", word_wrap=True)),
        border_style="grey42", padding=(1, 1)))
    t = Table(box=SIMPLE_HEAVY, expand=True, pad_edge=False)
    t.add_column(f"clean (no patch) — should STAY on `{cur_v}`", style="sea_green3",
                 ratio=1, overflow="fold")
    t.add_column(f"patched (move-on) — should jump to `{tgt_v}`", style="salmon1",
                 ratio=1, overflow="fold")
    t.add_row(escape(clean.strip()), escape(patched.strip()))
    rich_print(t); rich_print("")

decision heads 181 · PATCH_POS=-2 · 192 pairs on 'list[str]' · vector (64, 5120)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ml_s1_mul_566423  ·  datatype list[str]  ·  target_slot=1                                                       │
│ pivot on b   ->   move-on target c      (stay token 'b' · move-on token 'c')                                    │
│                                                                                                                 │
│   a. words has at least 3 entries   (already written)                                                           │
│   b. two of the words in words start with the same letter   <- CURRENT: pivot is mid-implementation             │
│   c. the words in words have more than 15 characters in total   <- TARGET: move-on should land here             │
│   d. all words in words have the same length                                                                    │
│   e. the longest word in words has at least 3 characters                                                        │
│   f. every word in words is entirely lowercase                                                                  │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = len(words)                                                                                              │
│     a = a >= 3                                                                                                  │
│     b = set(w[:1].lower() for w in words)                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (no patch) — should STAY on `b`                     patched (move-on) — should jump to `c`                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = len(b) < len(words)                                   c = sum(len(w) for w in words) > 15                     
     c = sum(len(w) for w in words)                            d = len(set(len(w) for w in words)) == 1            
     c = c > 15                                                e = max(len(w) for w in words) >= 3                 
     d = len(set(len(w) for w in words))                                                                           
     d                                                                                                            

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ml_s1_mul_688000  ·  datatype list[str]  ·  target_slot=1                                                       │
│ pivot on b   ->   move-on target c      (stay token 'b' · move-on token 'c')                                    │
│                                                                                                                 │
│   a. at least 1 words in words start with a vowel   (already written)                                           │
│   b. the words in words have more than 15 characters in total   <- CURRENT: pivot is mid-implementation         │
│   c. all words in words are distinct   <- TARGET: move-on should land here                                      │
│   d. every word in words is entirely lowercase                                                                  │
│   e. two of the words in words start with the same letter                                                       │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = sum(w != "" and w[:1].lower() in "aeiou" for w in words)                                                │
│     a = a >= 1                                                                                                  │
│     b = sum(len(w) for w in words)                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (no patch) — should STAY on `b`                     patched (move-on) — should jump to `c`                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = b > 15                                                c = len(words) == len(set(words))                       
     c = len(words) == len(set(words))                         d = all(w.islower() for w in words)                 
     d = all(w.islower() for w in words)                       e = len(set(w[:1] for w in words)) < len(words)     
     e = len(set(w[:1] for w in words))                        print(f"                                           

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ml_s1_mul_152727  ·  datatype list[str]  ·  target_slot=1                                                       │
│ pivot on b   ->   move-on target c      (stay token 'b' · move-on token 'c')                                    │
│                                                                                                                 │
│   a. the words in words are in alphabetical order   (already written)                                           │
│   b. two of the words in words start with the same letter   <- CURRENT: pivot is mid-implementation             │
│   c. the longest word in words has at least 3 characters   <- TARGET: move-on should land here                  │
│   d. at least 1 words in words start with a vowel                                                               │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = sorted(words)                                                                                           │
│     a = a == words                                                                                              │
│     b = set(w[:1].lower() for w in words)                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (no patch) — should STAY on `b`                     patched (move-on) — should jump to `c`                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = len(b) < len(words)                                   c = max(len(w) for w in words) >= 3                     
     c = max(len(w) for w in words) >= 3                       d = any(w[0].lower() in 'aeiou' for w in words)     
     d = any(w[0].lower() in 'aeiou' for w in words)           print(f"Condition 1: {a}, Condition                

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ml_s1_mul_008261  ·  datatype list[str]  ·  target_slot=1                                                       │
│ pivot on b   ->   move-on target c      (stay token 'b' · move-on token 'c')                                    │
│                                                                                                                 │
│   a. the longest word in words has at least 3 characters   (already written)                                    │
│   b. all words in words have the same length   <- CURRENT: pivot is mid-implementation                          │
│   c. every word in words is entirely lowercase   <- TARGET: move-on should land here                            │
│   d. all words in words are distinct                                                                            │
│   e. words has at least 4 entries                                                                               │
│   f. at least 1 words in words start with a vowel                                                               │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = max(len(w) for w in words)                                                                              │
│     a = a >= 3                                                                                                  │
│     b = set(len(w) for w in words)                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (no patch) — should STAY on `b`                     patched (move-on) — should jump to `c`                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = len(b) == 1                                           c = all(w.islower() for w in words)                     
     c = all(w.islower() for w in words)                       d = len(words) == len(set(words))                   
     d = len(set(words)) == len(words)                         e = len(words) >= 4                                 
     e = len(words) >= 4                                       f = any(w[0] in 'aeiou'                             
     f =                                                                                                          

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ ml_s1_mul_672786  ·  datatype list[str]  ·  target_slot=1                                                       │
│ pivot on b   ->   move-on target c      (stay token 'b' · move-on token 'c')                                    │
│                                                                                                                 │
│   a. every word in words is entirely lowercase   (already written)                                              │
│   b. words contains a word longer than 5 characters   <- CURRENT: pivot is mid-implementation                   │
│   c. all words in words are distinct   <- TARGET: move-on should land here                                      │
│   d. at least 3 words in words start with a vowel                                                               │
│   e. the longest word in words has at least 5 characters                                                        │
│   f. two of the words in words start with the same letter                                                       │
│                                                                                                                 │
│ prefill (pivot = end of the last line):                                                                         │
│                                                                                                                 │
│ def check_conditions(words: list[str]):                                                                         │
│     a = [w for w in words if w != w.lower()]                                                                    │
│     a = len(a) == 0                                                                                             │
│     b = [w for w in words if len(w) > 5]                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

clean (no patch) — should STAY on `b`                     patched (move-on) — should jump to `c`                  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
 b = len(b) > 0                                            c = len(words) == len(set(words))                       
     c = len(words) == len(set(words))                         d = [w for w in words if w[0] in 'aeiou']           
     d = [w for w in words if w[0] in 'aeiou']                 d = len(d) >= 3                                     
     d = len(d) >=                                             e = [len(w)                                        